# Pandas: limitations and alternative ecosystems

Lino Galiana  
2026-07-20

<div class="badge-container"><div class="badge-text">If you want to try the examples in this tutorial:</div><a href="https://github.com/linogaliana/python-datascientist-notebooks/blob/main/notebooks/en/manipulation/02_pandas_beyond.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/static/v1?logo=github&label=&message=View%20on%20GitHub&color=181717" alt="View on GitHub"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/vscode-python?autoLaunch=true&name=«02_pandas_beyond»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-vscode.sh»&init.personalInitArgs=«en/manipulation%2002_pandas_beyond»" class="badge" target="_blank" rel="noopener"><img src="https://custom-icon-badges.demolab.com/badge/SSP%20Cloud-Lancer_avec_VSCode-blue?logo=vsc&logoColor=white" alt="Onyxia"></a>
<a href="https://datalab.sspcloud.fr/launcher/ide/jupyter-python?autoLaunch=true&name=«02_pandas_beyond»&init.personalInit=«https%3A%2F%2Fraw.githubusercontent.com%2Flinogaliana%2Fpython-datascientist%2Fmain%2Fsspcloud%2Finit-jupyter.sh»&init.personalInitArgs=«en/manipulation%2002_pandas_beyond»" class="badge" target="_blank" rel="noopener"><img src="https://img.shields.io/badge/SSP%20Cloud-Lancer_avec_Jupyter-orange?logo=Jupyter&logoColor=orange" alt="Onyxia"></a>
<a href="https://colab.research.google.com/github/linogaliana/python-datascientist-notebooks-colab//en/blob/main//notebooks/en/manipulation/02_pandas_beyond.ipynb" class="badge" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://codespaces.new/linogaliana/python-datascientist-notebooks?quickstart=1&ref=main" class="badge" target="_blank" rel="noopener"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces"></a><br></div>

> **None**
>
> <div class="callout callout-style-default callout-note callout-titled">
> <div class="callout-header d-flex align-content-center">
> <div class="callout-icon-container">
> <i class="callout-icon"></i>
> </div>
> <div class="callout-title-container flex-fill">
> Note
> </div>
> </div>
> <div class="callout-body-container callout-body">
>
> Ceci est la version française 🇫🇷 de ce chapitre, pour voir la version anglaise rendez-vous sur <a href="https://pythonds.linogaliana.fr//home/runner/work/python-datascientist/python-datascientist/en/content/manipulation/02_pandas_beyond.qmd">le site du cours</a>.
>
> </div>
> </div>


<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Skills to be acquired by the end of this chapter
</div>
</div>
<div class="callout-body-container callout-body">

-   Know the main limitations of `Pandas` syntax;
-   Know how to chain operations more readably using pipes;
-   Know the main alternative packages to `Pandas` (`Polars`, `DuckDB`, `Spark`).

</div>
</div>

# 1. Introduction

The previous two chapters explored in depth the richness of the `Pandas` ecosystem, whether it was [associating data through joins](../../content/manipulation/02_pandas_joins.qmd) or [building descriptive statistics and reshaping data](../../content/manipulation/02_pandas_stats.qmd). `Pandas` is indispensable in the data scientist’s toolbox.

This short chapter takes a step back. Despite all its qualities, `Pandas` is not perfect: its syntax, inherited from more than fifteen years of history, has a few rough edges. We will also see that other, more recent, paradigms make it possible to go beyond `Pandas`, especially when it comes to handling larger volumes of data.

## 1.1 Data used in this chapter

To illustrate this chapter, we start again from the raw Ademe greenhouse gas emissions dataset, already used in the previous chapters:


In [ ]:
import numpy as np
import pandas as pd

url = "https://data.ademe.fr/data-fair/api/v1/datasets/igt-pouvoir-de-rechauffement-global/convert"
emissions = pd.read_csv(url)
emissions['dep'] = emissions["INSEE commune"].str[:2]

To obtain reproducible results, you can set the seed of the pseudo-random number generator.


In [ ]:
np.random.seed(123)

# 2. `Pandas` in a chain of operations

Generally, in a project, data cleaning will consist of a series of methods applied to a `DataFrame` or a `Series` when working exclusively on a single column. In other words, what is usually expected when working with `Pandas` is to have a chain that takes a `DataFrame` as input and outputs the same `DataFrame` enriched or an aggregated version of it.

This way of proceeding is at the heart of the `dplyr` syntax in `R` but is not necessarily native in `Pandas` depending on the operations you want to implement. Indeed, the natural way to update a dataframe in `Pandas` often involves syntax like:


In [ ]:
import numpy as np
import pandas as pd

data = [[8000, 1000], [9500, np.nan], [5000, 2000]]
df = pd.DataFrame(data, columns=['salaire', 'autre_info'])
df['salaire_net'] = df['salaire']*0.8

In `SQL` you could directly update your database with the new column:

``` sql
SELECT *, salaire*0.8 AS salaire_net FROM df
```

The `tidyverse` ecosystem in `R`, the equivalent of `Pandas`, works according to the same logic as SQL table updates. Indeed, you would use the following command with `dplyr`:

``` r
df %>% mutate(salaire_net = salaire*0.8)
```

Technically, you could do this with an `assign` in `Pandas`:


In [ ]:
df = df.drop("salaire_net", axis = "columns")
df = df.assign(salaire_net = lambda s: s['salaire']*0.8)

However, this `assign` syntax is not very natural. It requires passing a lambda function that expects a `DataFrame` as input where you would want a column. So, it is not really a readable and practical syntax.

It is nevertheless possible to chain operations on datasets using [pipes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html). These follow the same philosophy as `dplyr`, itself inspired by the Linux pipe. This approach will make the code more readable by defining functions that perform operations on one or more columns of a DataFrame. The first argument to the function is the `DataFrame`, the others are those controlling its behavior:


In [ ]:
def calcul_salaire_net(df: pd.DataFrame, col: str, taux: float = 0.8):
  df["salaire_net"] = df[col]*taux
  return df

This transforms our production chain into:


In [ ]:
(
  df
  .pipe(calcul_salaire_net, "salaire")
)

## 2.1 Some limitations regarding `Pandas` syntax

There is a before and after `Pandas` in data analysis with `Python`. Without this incredibly practical package, Python, despite all its strengths, would have struggled to establish itself in the data analysis landscape. However, while `Pandas` offers a coherent syntax in many aspects, it is not perfect either. More recent data analysis paradigms in `Python` sometimes aim to correct these syntactical imperfections.

Among the most annoying points in everyday use is the need to regularly perform `reset_index` when building descriptive statistics. Indeed, it can be dangerous to keep indices that are not well controlled because, if we are not careful during the merge phases, they can be misused by `Pandas` to join data, leading to surprises.

`Pandas` is extremely well-designed for restructuring data from long to wide format or vice versa. However, this is not the only way to restructure a dataset that we might want to implement. It often happens that we want to compare the value of an observation to that of a group to which it belongs. This is particularly useful in anomaly analysis, outlier detection, or fraud investigation. Natively, in `Pandas`, you need to build an aggregate statistic by group and then merge it back to the initial data using the group variable. This is somewhat tedious:


In [ ]:
emissions_moyennes = emissions.groupby("dep").agg({"Agriculture": "mean"}).reset_index()
emissions_enrichies = (
  emissions
  .merge(emissions_moyennes, on = "dep", suffixes = ['', '_moyenne_dep'])
)
emissions_enrichies['relatives'] = emissions_enrichies["Agriculture"]/emissions_enrichies["Agriculture_moyenne_dep"]
emissions_enrichies.head()

In the `tidyverse`, this two-step operation could be done in a single step, which is more convenient:

``` r
emissions %>%
  group_by(dep) %>%
  mutate(relatives = Agriculture/mean(Agriculture))
```

This isn’t too bad, but it does make `Pandas` processing chains longer and therefore increases the maintenance burden to keep them running over time.

More generally, `Pandas` processing chains can be quite verbose because it is often necessary to redefine the `DataFrame` rather than just the columns. For example, to filter rows and columns, you have to:


In [ ]:
(
  emissions
  .loc[
    (emissions["dep"] == "12") & (emissions["Routier"]>500), ['INSEE commune', 'Commune']
  ]
  .head(5)
)

In SQL, you could simply refer to the columns in the filter:

``` sql
SELECT "INSEE commune", 'Commune'
FROM emissions
WHERE dep=="12" AND Routier>500
```

In the `tidyverse` (R), you could also do this simply:

``` r
df %>%
  filter(dep=="12", Routier>500) %>%
  select(`INSEE commune`, `Commune`)
```

# 3. Other paradigms

Despite all the limitations we have just mentioned, and the alternative solutions we will present, `Pandas` remains the central package of the data ecosystem with `Python`. In the following chapters, we will see its native integration with the `Scikit` ecosystem for machine learning or the extension of `Pandas` to spatial data with `GeoPandas`.

Other technical solutions that we will discuss here may be relevant if you want to handle large volumes of data or if you want to use alternative syntaxes.

The main alternatives to `Pandas` are [`Polars`](https://pola.rs/), [`DuckDB`](https://duckdb.org/), and [`Spark`](https://spark.apache.org/docs/latest/api/python/index.html). There is also [`Dask`](https://www.dask.org/), a library for parallelizing `Pandas` operations.

## 3.1 `Polars`

`Polars` is certainly the paradigm most inspired by `Pandas`, even in the choice of name. The first fundamental difference lies in the internal layers used. `Polars` relies on the `Rust` implementation of `Arrow`, whereas `Pandas` relies on `Numpy`, which results in performance loss. This allows `Polars` to be more efficient on large volumes of data, especially since many operations are parallelized and rely on lazy evaluation, a programming principle that optimizes queries for logical rather than defined execution order.

Another strength of `Polars` is its more coherent syntax, benefiting from over fifteen years of `Pandas` existence and almost a decade of `dplyr` (the data manipulation package within the `R` tidyverse paradigm). To take the previous example, there is no longer a need to force the reference to the DataFrame; in an execution chain, all subsequent references will be made with respect to the initial DataFrame.


In [ ]:
import polars as pl
emissions_polars = pl.from_pandas(emissions)
(
  emissions_polars
  .filter(pl.col("dep") == "12", pl.col("Routier") > 500)
  .select('INSEE commune', 'Commune')
  .head(5)
)

To learn about `Polars`, many online resources are available, including [this notebook](https://github.com/InseeFrLab/ssphub/blob/main/post/polars/polars-tuto.ipynb) built for the public statistics data scientists network.

## 3.2 `DuckDB`

`DuckDB` is the newcomer in the data analysis ecosystem, pushing the limits of data processing with `Python` without resorting to big data tools like `Spark`. `DuckDB` epitomizes a new paradigm, the [“Big data is dead”](https://motherduck.com/blog/big-data-is-dead/) paradigm, where large data volumes can be processed without imposing infrastructures.

Besides its great efficiency, as `DuckDB` can handle data volumes larger than the computer or server’s RAM, it offers the advantage of a uniform syntax across languages that call `DuckDB` (`Python`, `R`, `C++`, or `Javascript`). `DuckDB` favors SQL syntax for data processing with many pre-implemented functions to simplify certain data transformations (e.g., for [text data](https://duckdb.org/docs/sql/functions/char.html), [time data](https://duckdb.org/docs/sql/functions/time), etc.).

Compared to other SQL-based systems like [`PostGreSQL`](https://www.postgresql.org/), `DuckDB` is very simple to install, as it is just a `Python` library, whereas many tools like `PostGreSQL` require an appropriate infrastructure.

To reuse the previous example, we can directly use the SQL code mentioned earlier.


In [ ]:
import duckdb
duckdb.sql(
  """
  SELECT "INSEE commune", "Commune"
  FROM emissions
  WHERE dep=='12' AND Routier>500
  LIMIT 5
  """)

┌───────────────┬─────────────────────┐
│ INSEE commune │       Commune       │
│    varchar    │       varchar       │
├───────────────┼─────────────────────┤
│ 12001         │ AGEN-D'AVEYRON      │
│ 12002         │ AGUESSAC            │
│ 12006         │ ALRANCE             │
│ 12007         │ AMBEYRAC            │
│ 12008         │ ANGLARS-SAINT-FELIX │
└───────────────┴─────────────────────┘

Here, the clause `FROM emissions` comes from the fact that we can directly execute SQL from a `Pandas` object via `DuckDB`. If we read directly in the query, it gets slightly more complex, but the logic remains the same.


In [ ]:
import duckdb
duckdb.sql(
  f"""
  SELECT "INSEE commune", "Commune"
  FROM read_csv_auto("{url}")
  WHERE
    substring("INSEE commune",1,2)=='12'
    AND
    Routier>500
  LIMIT 5
  """)

┌───────────────┬─────────────────────┐
│ INSEE commune │       Commune       │
│    varchar    │       varchar       │
├───────────────┼─────────────────────┤
│ 12001         │ AGEN-D'AVEYRON      │
│ 12002         │ AGUESSAC            │
│ 12006         │ ALRANCE             │
│ 12007         │ AMBEYRAC            │
│ 12008         │ ANGLARS-SAINT-FELIX │
└───────────────┴─────────────────────┘

The rendering of the DataFrame is slightly different from `Pandas` because, like `Polars` and many large data processing systems, `DuckDB` relies on lazy evaluation and thus only displays a sample of data. `DuckDB` and `Polars` are also well integrated with each other. You can run SQL on a `Polars` object via `DuckDB` or apply `Polars` functions to an initially read `DuckDB` object.

One of the interests of `DuckDB` is its excellent integration with the `Parquet` ecosystem, the already mentioned data format that is becoming a standard in data sharing (for example, it is the cornerstone of data sharing on the HuggingFace platform). To learn more about `DuckDB` and discover its usefulness for reading data from the French population census, you can check out [this blog post](https://ssphub.netlify.app/post/parquetrp/).

## 3.3 `Spark`

`DuckDB` has pushed the boundaries of big data, which can be defined as the volume of data that can no longer be processed on a single machine without implementing a parallelization strategy.

Nevertheless, for very large data volumes, `Python` is well-equipped with the [`PySpark`](https://spark.apache.org/docs/latest/api/python/index.html) library. This is a Python API for the Spark language, a big data language based on Scala. This paradigm is built on the idea that `Python` users access it via clusters with many nodes to process data in parallel. The data will be read in blocks, processed in parallel depending on the number of parallel nodes. The `Spark` DataFrame API has a syntax close to previous paradigms with more complex engineering in the background related to native parallelization.
